# Agent高级用法-ToolStrategy 结构化输出
## 1. 结构化输出schema参数（4种模式：pydantic、typedict、jsonSchema、@dataClass）

### 1.1 Pydantic类型

In [4]:
from dataclasses import dataclass

from pydantic import Field, BaseModel
from typing import Literal, TypedDict
from langchain_core.tools import tool
from langchain.agents.structured_output import ToolStrategy
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
import os

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from rich import print as rprint

# 加载配置文件，存在相同key采用当前覆盖
load_dotenv(override=True)

# 具体模型的key和url
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_API_BASE   = os.getenv("DEEPSEEK_BASE_URL")
DEEPSEEK_MODEL_NAME   = os.getenv("DEEPSEEK_MODEL")

# 定义工具
@tool(parse_docstring=True)
def search_customer_database(query: str) -> str:
    """
    在客户数据库中搜索信息

    Args:
    query (str): 客户查询字符串，例如 "张三" 或 "李四"

    Returns:
    str: 客户记录字符串，包含客户姓名、等级、最近购买日期和累计消费
    """
    # 模拟数据库查询结果
    if "张三" in query.lower():
        return "客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000"
    elif "李四" in query.lower():
        return "客户记录：李四，普通客户，最近购买日期：2025-12-20，累计消费：$3,200"
    else:
        return f"关于客户{query}，无记录"


@tool(parse_docstring=True)
def send_email(customer: str) -> str:
    """
    发送感谢邮件

    Args:
    customer (str): 客户名称，例如 "张三" 或 "李四"

    Returns:
    str: 确认消息，包含已发送的客户名称
    """

    return f"已向 {customer} 发送感谢邮件"


# 定义Pydantic Schema
class CustomerAnalysis(BaseModel):
    """客户分析报告"""
    customer_name: str = Field(None, description="客户姓名")
    customer_tier: Literal["潜在客户", "普通客户", "VIP客户", "流失风险"] =Field("潜在客户",description="客户等级,只能是潜在客户、普通客户、VIP客户或流失风险")
    recent_activity: str = Field(None, description="最近活动")
    spending_level: Literal["低", "中", "高"] = Field(None, description="消费水平")
    send_email: bool = Field(False, description="是否已发送感谢邮件")

model = init_chat_model(
    model=DEEPSEEK_MODEL_NAME,
    model_provider="deepseek",
    api_key = DEEPSEEK_API_KEY,
    base_url = DEEPSEEK_API_BASE,
    extra_body={"thinking":{"type":"disabled"}}

)
# 创建agent
agent = create_agent(
    model=model,
    tools=[send_email,search_customer_database],
    response_format=ToolStrategy(CustomerAnalysis),
    system_prompt=SystemMessage(content=""
        "请分析指定客户的情况："
        "1. 先搜索客户数据库了解最新情况 "
        "2. 如果是VIP客户，则发送感谢邮件 "
        "3. 基于搜索结果生成结构化分析报告 "
        "4. 如果用户提问与客户记录无关或找不到客户信息，则返回空对象，不发送感谢邮件"
    )
)


# 执行分析
result = agent.invoke({
"messages": [{"role": "user", "content": "请分析客户张三"}]
# "messages": [{"role": "user","content": "请分析客户李四"}]
# "messages": [{"role": "user","content": "请分析客户王五"}]
# "messages": [{"role": "user","content": "今天天气如何"}]
})
# 处理结果
rprint(result)
# if "structured_response" in result:
#     analysis = result["structured_response"]
#     print(analysis)

{
    'messages': [
        HumanMessage(
            content='请分析客户张三',
            additional_kwargs={},
            response_metadata={},
            id='37e99e44-d1cf-4f42-90bf-5005d93719fb'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 39,
                    'prompt_tokens': 622,
                    'total_tokens': 661,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 512
                    },
                    'prompt_cache_hit_tokens': 512,
                    'prompt_cache_miss_tokens': 110
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': 'b0e6c81a-d931-45d9-a4b7-f75b241fc51a',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019ffa90-51b1-79d3-aa64-6470bbdbccd0-0',
            tool_calls=[
                {
                    'name': 'search_customer_database',
                    'args': {'query': '张三'},
                    'id': 'call_00_azs4P8duS8vYhOkzs1Wk5352',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 622,
                'output_tokens': 39,
                'total_tokens': 661,
                'input_token_details': {'cache_read': 512},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000',
            name='search_customer_database',
            id='3baf0221-0e61-4cf5-afcd-e7588065134b',
            tool_call_id='call_00_azs4P8duS8vYhOkzs1Wk5352'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 149,
                    'prompt_tokens': 706,
                    'total_tokens': 855,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 640
                    },
                    'prompt_cache_hit_tokens': 640,
                    'prompt_cache_miss_tokens': 66
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': '4f2bc88f-7d67-4ff3-a84c-5bbdbeb51e50',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019ffa90-559a-7a91-97c0-7cc5353089e4-0',
            tool_calls=[
                {
                    'name': 'send_email',
                    'args': {'customer': '张三'},
                    'id': 'call_00_ji0izLELuzrEmujZj0aQ6774',
                    'type': 'tool_call'
                },
                {
                    'name': 'CustomerAnalysis',
                    'args': {
                        'customer_name': '张三',
                        'customer_tier': 'VIP客户',
                        'recent_activity': '最近购买日期：2026-01-15',
                        'spending_level': '高',
                        'send_email': True
                    },
                    'id': 'call_01_UPcrpi1dKrlTtc3bCq1f8359',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_me

### 1.2 TypeDict类型

In [7]:

from typing import Literal, Optional
from langchain_core.tools import tool
from langchain.agents.structured_output import ToolStrategy
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from typing import TypedDict,Annotated
import os

from dotenv import load_dotenv
from langchain_core.messages import  SystemMessage
from rich import print as rprint

# 加载配置文件，存在相同key采用当前覆盖
load_dotenv(override=True)

# 具体模型的key和url
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_API_BASE   = os.getenv("DEEPSEEK_BASE_URL")
DEEPSEEK_MODEL_NAME   = os.getenv("DEEPSEEK_MODEL")

# 定义工具
@tool(parse_docstring=True)
def search_customer_database(query: str) -> str:
    """
    在客户数据库中搜索信息

    Args:
    query (str): 客户查询字符串，例如 "张三" 或 "李四"

    Returns:
    str: 客户记录字符串，包含客户姓名、等级、最近购买日期和累计消费
    """
    # 模拟数据库查询结果
    if "张三" in query.lower():
        return "客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000"
    elif "李四" in query.lower():
        return "客户记录：李四，普通客户，最近购买日期：2025-12-20，累计消费：$3,200"
    else:
        return f"关于客户{query}，无记录"


@tool(parse_docstring=True)
def send_email(customer: str) -> str:
    """
    发送感谢邮件

    Args:
    customer (str): 客户名称，例如 "张三" 或 "李四"

    Returns:
    str: 确认消息，包含已发送的客户名称
    """

    return f"已向 {customer} 发送感谢邮件"


# 定义Pydantic Schema
class CustomerAnalysis(TypedDict):
    """客户分析报告"""
    customer_name : Annotated[Optional[str],None, "客户姓名"]
    customer_tier : Annotated[Optional[Literal["潜在客户", "普通客户", "VIP客户", "流失风险"]],"客户等级,只能是潜在客户、普通客户、VIP客户或流失风险"]
    recent_activity : Annotated[str,None, "最近活动"]
    spending_level : Annotated[Literal["低", "中", "高"],None, "消费水平"]
    send_email : Annotated[bool,False, "是否已发送感谢邮件"]

model = init_chat_model(
    model=DEEPSEEK_MODEL_NAME,
    model_provider="deepseek",
    api_key = DEEPSEEK_API_KEY,
    base_url = DEEPSEEK_API_BASE,
    extra_body={"thinking":{"type":"disabled"}}

)
# 创建agent
agent = create_agent(
    model=model,
    tools=[send_email,search_customer_database],
    response_format=ToolStrategy(CustomerAnalysis),
    system_prompt=SystemMessage(content=""
        "请分析指定客户的情况："
        "1. 先搜索客户数据库了解最新情况 "
        "2. 如果是VIP客户，则发送感谢邮件 "
        "3. 基于搜索结果生成结构化分析报告 "
        "4. 如果用户提问与客户记录无关或找不到客户信息，则返回空对象，不发送感谢邮件"
    )
)


# 执行分析
result = agent.invoke({
"messages": [{"role": "user", "content": "请分析客户张三"}]
# "messages": [{"role": "user","content": "请分析客户李四"}]
# "messages": [{"role": "user","content": "请分析客户王五"}]
# "messages": [{"role": "user","content": "今天天气如何"}]
})
# 处理结果
rprint(result)
# if "structured_response" in result:
#     analysis = result["structured_response"]
#     print(analysis)

{
    'messages': [
        HumanMessage(
            content='请分析客户张三',
            additional_kwargs={},
            response_metadata={},
            id='b8ea44c4-c7bb-4d11-9674-a795c8d41ca1'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 39,
                    'prompt_tokens': 597,
                    'total_tokens': 636,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 0
                    },
                    'prompt_cache_hit_tokens': 0,
                    'prompt_cache_miss_tokens': 597
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': 'b3b02f59-d073-4d3d-8d35-8995348a14f3',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019ffa9a-ee74-7b50-8c93-16b5c72a1295-0',
            tool_calls=[
                {
                    'name': 'search_customer_database',
                    'args': {'query': '张三'},
                    'id': 'call_00_4sVc7H5GZspnuArm3jqs8231',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 597,
                'output_tokens': 39,
                'total_tokens': 636,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000',
            name='search_customer_database',
            id='2682c6bd-a735-42e5-ac17-84ff1cb84993',
            tool_call_id='call_00_4sVc7H5GZspnuArm3jqs8231'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 149,
                    'prompt_tokens': 681,
                    'total_tokens': 830,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 512
                    },
                    'prompt_cache_hit_tokens': 512,
                    'prompt_cache_miss_tokens': 169
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': '106b07c4-9a41-4c35-aa9d-50b43fcd20cf',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019ffa9a-f2e2-7533-903e-d8550441ba36-0',
            tool_calls=[
                {
                    'name': 'send_email',
                    'args': {'customer': '张三'},
                    'id': 'call_00_PZ4FmgOJvmeOfCzoebxp6205',
                    'type': 'tool_call'
                },
                {
                    'name': 'CustomerAnalysis',
                    'args': {
                        'customer_name': '张三',
                        'customer_tier': 'VIP客户',
                        'recent_activity': '最近购买日期：2026-01-15',
                        'spending_level': '高',
                        'send_email': True
                    },
                    'id': 'call_01_TTbV0mIV3vmMTIN1g10h1808',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadat

### 1.3 jsonSchema 不推荐

### 1.4 @dataclass

In [10]:

from langchain_core.tools import tool
from langchain.agents.structured_output import ToolStrategy
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
import os
from dataclasses import dataclass

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from rich import print as rprint

# 加载配置文件，存在相同key采用当前覆盖
load_dotenv(override=True)

# 具体模型的key和url
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_API_BASE   = os.getenv("DEEPSEEK_BASE_URL")
DEEPSEEK_MODEL_NAME   = os.getenv("DEEPSEEK_MODEL")

# 定义工具
@tool(parse_docstring=True)
def search_customer_database(query: str) -> str:
    """
    在客户数据库中搜索信息

    Args:
    query (str): 客户查询字符串，例如 "张三" 或 "李四"

    Returns:
    str: 客户记录字符串，包含客户姓名、等级、最近购买日期和累计消费
    """
    # 模拟数据库查询结果
    if "张三" in query.lower():
        return "客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000"
    elif "李四" in query.lower():
        return "客户记录：李四，普通客户，最近购买日期：2025-12-20，累计消费：$3,200"
    else:
        return f"关于客户{query}，无记录"


@tool(parse_docstring=True)
def send_email(customer: str) -> str:
    """
    发送感谢邮件

    Args:
    customer (str): 客户名称，例如 "张三" 或 "李四"

    Returns:
    str: 确认消息，包含已发送的客户名称
    """

    return f"已向 {customer} 发送感谢邮件"


@dataclass
class CustomerAnalysis():
    """客户分析报告"""
    customer_name: str = Field(description="客户姓名")
    customer_tier: Literal["潜在客户", "普通客户", "VIP客户", "流失风险"] = Field(description="客户等级,只能是潜在客户、普通客户、VIP客户或流失风险")
    recent_activity: str = Field( description="最近活动")
    spending_level: Literal["低", "中", "高"] = Field(description="消费水平")
    send_email: bool = Field(False, description="是否已发送感谢邮件")

model = init_chat_model(
    model=DEEPSEEK_MODEL_NAME,
    model_provider="deepseek",
    api_key = DEEPSEEK_API_KEY,
    base_url = DEEPSEEK_API_BASE,
    extra_body={"thinking":{"type":"disabled"}}

)
# 创建agent
agent = create_agent(
    model=model,
    tools=[send_email,search_customer_database],
    response_format=ToolStrategy(CustomerAnalysis),
    system_prompt=SystemMessage(content=""
        "请分析指定客户的情况："
        "1. 先搜索客户数据库了解最新情况 "
        "2. 如果是VIP客户，则发送感谢邮件 "
        "3. 基于搜索结果生成结构化分析报告 "
        "4. 如果用户提问与客户记录无关或找不到客户信息，则返回空对象，不发送感谢邮件"
    )
)


# 执行分析
result = agent.invoke({
"messages": [{"role": "user", "content": "请分析客户张三"}]
# "messages": [{"role": "user","content": "请分析客户李四"}]
# "messages": [{"role": "user","content": "请分析客户王五"}]
# "messages": [{"role": "user","content": "今天天气如何"}]
})
# 处理结果
rprint(result)
# if "structured_response" in result:
#     analysis = result["structured_response"]
#     print(analysis)

{
    'messages': [
        HumanMessage(
            content='请分析客户张三',
            additional_kwargs={},
            response_metadata={},
            id='83295481-e754-4826-b9c8-7b2769c7802d'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 39,
                    'prompt_tokens': 623,
                    'total_tokens': 662,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 384
                    },
                    'prompt_cache_hit_tokens': 384,
                    'prompt_cache_miss_tokens': 239
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': '1c5c23b9-9b1e-4e0f-a539-6d595e10b999',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019ffaa0-ba82-7da1-800b-45a9fab40323-0',
            tool_calls=[
                {
                    'name': 'search_customer_database',
                    'args': {'query': '张三'},
                    'id': 'call_00_d4BYq0UdOsTYPvhcgd297122',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 623,
                'output_tokens': 39,
                'total_tokens': 662,
                'input_token_details': {'cache_read': 384},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000',
            name='search_customer_database',
            id='436be385-6852-4d00-b4a8-b8e2495787ee',
            tool_call_id='call_00_d4BYq0UdOsTYPvhcgd297122'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 36,
                    'prompt_tokens': 707,
                    'total_tokens': 743,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 640
                    },
                    'prompt_cache_hit_tokens': 640,
                    'prompt_cache_miss_tokens': 67
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': '628cd17f-a134-4278-8e3a-dd35cd736dc2',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019ffaa0-bdf5-7611-a4cf-80f5a57d64b9-0',
            tool_calls=[
                {
                    'name': 'send_email',
                    'args': {'customer': '张三'},
                    'id': 'call_00_jAULXrDU0Esaky2PHzh15968',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 707,
                'output_tokens': 36,
                'total_tokens': 743,
                'input_token_details': {'cache_read': 640},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='已向 张三 发送感谢邮件',
            name='send_email',
            id='93c34f52-9144-4c86-b9e6-eb285647c0b5',
            tool_call_id='call_00_jAULXrDU0Esaky2PHzh15968'
        ),
        AIMessage(
           

## 2.0 多schema联合模式 ToolStrategy允许指定多个类型，用Union[类型1，类型2]写法

In [13]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.messages import HumanMessage
class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")


class EventInfo(BaseModel):
    """事件详情"""
    event_name: str = Field(description="事件名称")
    date: str = Field(description="事件发生日期")


class Person(BaseModel):
    """学生信息"""
    stu_name: str = Field(description="学生名称")
    stu_course: str = Field(description="学生成绩")


agent = create_agent(
model=model,
response_format=ToolStrategy(
    Union[ContactInfo, EventInfo]
    )
)
response = agent.invoke(
{
"messages": [
HumanMessage("从这段话中抽取结构化信息：小明的邮箱地址为：shkstart@atguigu.com，手机号：12345678912")
]
}
)
for msg in response["messages"]:
    msg.pretty_print()
    print(response["structured_response"])

================================ Human Message =================================

从这段话中抽取结构化信息：小明的邮箱地址为：shkstart@atguigu.com，手机号：12345678912
name='小明' email='shkstart@atguigu.com' phone='12345678912'
================================== Ai Message ==================================
Tool Calls:
  ContactInfo (call_00_PnEp3dV4urwgmOe7Tdnw0193)
 Call ID: call_00_PnEp3dV4urwgmOe7Tdnw0193
  Args:
    name: 小明
    email: shkstart@atguigu.com
    phone: 12345678912
name='小明' email='shkstart@atguigu.com' phone='12345678912'
================================= Tool Message =================================
Name: ContactInfo

Returning structured response: name='小明' email='shkstart@atguigu.com' phone='12345678912'
name='小明' email='shkstart@atguigu.com' phone='12345678912'


In [12]:
response = agent.invoke(
{
"messages": [
HumanMessage("从这段话中抽取结构化信息：2026年高考报名人数突破1200万")
]
}
)
for msg in response["messages"]:
    msg.pretty_print()
    print(response["structured_response"])

================================ Human Message =================================

从这段话中抽取结构化信息：2026年高考报名人数突破1200万
event_name='高考报名' date='2026'
================================== Ai Message ==================================
Tool Calls:
  EventInfo (call_00_FxZPYf8bbMGLbr5YDWT38469)
 Call ID: call_00_FxZPYf8bbMGLbr5YDWT38469
  Args:
    event_name: 高考报名
    date: 2026
event_name='高考报名' date='2026'
================================= Tool Message =================================
Name: EventInfo

Returning structured response: event_name='高考报名' date='2026'
event_name='高考报名' date='2026'


## 3.0 ToolStrategy的主要参数信息（tool_message_content:指定工具content的内容；handle_errors:模型解析参数时异常处理）

### 3.1 tool_message_content案例

In [14]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.messages import HumanMessage
class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")


class EventInfo(BaseModel):
    """事件详情"""
    event_name: str = Field(description="事件名称")
    date: str = Field(description="事件发生日期")


class Person(BaseModel):
    """学生信息"""
    stu_name: str = Field(description="学生名称")
    stu_course: str = Field(description="学生成绩")


agent = create_agent(
model=model,
response_format=ToolStrategy(
    Union[ContactInfo, EventInfo],
    tool_message_content="信息提取成功"
    )
)
response = agent.invoke(
{
"messages": [
HumanMessage("从这段话中抽取结构化信息：小明的邮箱地址为：shkstart@atguigu.com，手机号：12345678912")
]
}
)
rprint(response)

{
    'messages': [
        HumanMessage(
            content='从这段话中抽取结构化信息：小明的邮箱地址为：shkstart@atguigu.com，手机号：12345678912',
            additional_kwargs={},
            response_metadata={},
            id='746abb67-5632-4a6f-aaad-f6fad0c6e982'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 78,
                    'prompt_tokens': 424,
                    'total_tokens': 502,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 384
                    },
                    'prompt_cache_hit_tokens': 384,
                    'prompt_cache_miss_tokens': 40
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': '2cd63498-025a-46bc-a203-f4981f5fca7a',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019ffab1-1303-7d12-b5b4-9ee62b01d129-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '小明', 'email': 'shkstart@atguigu.com', 'phone': '12345678912'},
                    'id': 'call_00_0el4meLWsqyyjfJDBC1O6383',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 424,
                'output_tokens': 78,
                'total_tokens': 502,
                'input_token_details': {'cache_read': 384},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='信息提取成功',
            name='ContactInfo',
            id='98282154-c8d3-4257-ac17-b0977cda6137',
            tool_call_id='call_00_0el4meLWsqyyjfJDBC1O6383'
        )
    ],
    'structured_response': ContactInfo(name='小明', email='shkstart@atguigu.com', phone='12345678912')
}

### 3.2 handle_errors参数（值 True：langchain默认值，捕获异常，使用内置 错误消息模版 提示重试；False：关闭重试机制，任何异常都跑出；自定义字符串：为用户友好提示；callable：自定义函数处理异常

#### 3.2.1 True/False 设置为True会自动重试

In [15]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.messages import HumanMessage
class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")


class EventInfo(BaseModel):
    """事件详情"""
    event_name: str = Field(description="事件名称")
    date: str = Field(description="事件发生日期")


agent = create_agent(
model=model,
response_format=ToolStrategy(
    Union[ContactInfo, EventInfo],
    tool_message_content="信息提取成功",
    handle_errors=True
    )
)
response = agent.invoke(
{
"messages": [
HumanMessage("请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：2026-07-15")
]
}
)
rprint(response)

{
    'messages': [
        HumanMessage(
            content='请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：
2026-07-15',
            additional_kwargs={},
            response_metadata={},
            id='784c50a0-2b75-4dfc-988b-1cab1e5bea86'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 126,
                    'prompt_tokens': 433,
                    'total_tokens': 559,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 384
                    },
                    'prompt_cache_hit_tokens': 384,
                    'prompt_cache_miss_tokens': 49
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': 'd2acf43a-a0ca-4bba-ad69-f16e91114534',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019ffab5-f1a6-7f51-af8c-da9795ed8f37-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '张三', 'email': 'zhang3@atguigu.com', 'phone': ''},
                    'id': 'call_00_E8ZDi0vUm2tMfpw9oVV25891',
                    'type': 'tool_call'
                },
                {
                    'name': 'EventInfo',
                    'args': {'event_name': '公司年会', 'date': '2026-07-15'},
                    'id': 'call_01_OMR3cAMYgfBA2nNQMYLZ1491',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 433,
                'output_tokens': 126,
                'total_tokens': 559,
                'input_token_details': {'cache_read': 384},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='Error: Model incorrectly returned multiple structured responses (ContactInfo, EventInfo) when 
only one is expected.\n Please fix your mistakes.',
            name='ContactInfo',
            id='320fc011-a9b9-4536-ae6e-4b4b2a08a233',
            tool_call_id='call_00_E8ZDi0vUm2tMfpw9oVV25891'
        ),
        ToolMessage(
            content='Error: Model incorrectly returned multiple structured responses (ContactInfo, EventInfo) when 
only one is expected.\n Please fix your mistakes.',
            name='EventInfo',
            id='55afb659-a43a-45ae-9b68-a9d68499369c',
            tool_call_id='call_01_OMR3cAMYgfBA2nNQMYLZ1491'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 73,
                    'prompt_tokens': 636,
                    'total_tokens': 709,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 512
                    },
                    'prompt_cache_hit_tokens': 512,
                    'prompt_cache_miss_tokens': 124
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': 'd2e3cf23-3a3f-4b35-9955-7d787dd96287',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019ffab5-f774-7ca1-84c7-da360421938c-0',
            tool_calls=[
            

设置为False 直接抛出异常

In [16]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.messages import HumanMessage
class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")


class EventInfo(BaseModel):
    """事件详情"""
    event_name: str = Field(description="事件名称")
    date: str = Field(description="事件发生日期")


agent = create_agent(
model=model,
response_format=ToolStrategy(
    Union[ContactInfo, EventInfo],
    tool_message_content="信息提取成功",
    handle_errors=False
    )
)
response = agent.invoke(
{
"messages": [
HumanMessage("请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：2026-07-15")
]
}
)
rprint(response)

MultipleStructuredOutputsError: Model incorrectly returned multiple structured responses (ContactInfo, EventInfo) when only one is expected.

#### 3.2.2 设置指定异常 langchain会拦截重试

In [19]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy, MultipleStructuredOutputsError, \
    StructuredOutputValidationError
from langchain.messages import HumanMessage
class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")


class EventInfo(BaseModel):
    """事件详情"""
    event_name: str = Field(description="事件名称")
    date: str = Field(description="事件发生日期")


agent = create_agent(
model=model,
response_format=ToolStrategy(
    Union[ContactInfo, EventInfo],
    tool_message_content="信息提取成功",
    handle_errors=(MultipleStructuredOutputsError,StructuredOutputValidationError)
    )
)
response = agent.invoke(
{
"messages": [
HumanMessage("请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：2026-07-15")
]
}
)
rprint(response)

{
    'messages': [
        HumanMessage(
            content='请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：
2026-07-15',
            additional_kwargs={},
            response_metadata={},
            id='6fa60d43-799a-41e2-9468-92f83a867842'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 126,
                    'prompt_tokens': 433,
                    'total_tokens': 559,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 384
                    },
                    'prompt_cache_hit_tokens': 384,
                    'prompt_cache_miss_tokens': 49
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': 'b6820a2c-1da5-48fa-a6d1-f3c2851fbe6e',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019ffabb-ad85-7b50-8a49-297396fdd1d2-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '张三', 'email': 'zhang3@atguigu.com', 'phone': ''},
                    'id': 'call_00_kEGK5F4wwqQcT8cybFIA2045',
                    'type': 'tool_call'
                },
                {
                    'name': 'EventInfo',
                    'args': {'event_name': '公司年会', 'date': '2026-07-15'},
                    'id': 'call_01_cIsLy914Vea3Crkezb743941',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 433,
                'output_tokens': 126,
                'total_tokens': 559,
                'input_token_details': {'cache_read': 384},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='Error: Model incorrectly returned multiple structured responses (ContactInfo, EventInfo) when 
only one is expected.\n Please fix your mistakes.',
            name='ContactInfo',
            id='f01f8b27-4609-44c0-b2bf-a807ff86dea4',
            tool_call_id='call_00_kEGK5F4wwqQcT8cybFIA2045'
        ),
        ToolMessage(
            content='Error: Model incorrectly returned multiple structured responses (ContactInfo, EventInfo) when 
only one is expected.\n Please fix your mistakes.',
            name='EventInfo',
            id='b403cfbd-8242-4f10-9fb4-17dc874426b0',
            tool_call_id='call_01_cIsLy914Vea3Crkezb743941'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 73,
                    'prompt_tokens': 636,
                    'total_tokens': 709,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 512
                    },
                    'prompt_cache_hit_tokens': 512,
                    'prompt_cache_miss_tokens': 124
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': '7c65d9f7-9c51-4376-b40b-af4bc937c417',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019ffabb-b232-7e10-a917-0ade987be72a-0',
            tool_calls=[
            

#### 3.2.3 自定义异常

In [20]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy, MultipleStructuredOutputsError, \
    StructuredOutputValidationError
from langchain.messages import HumanMessage
class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")


class EventInfo(BaseModel):
    """事件详情"""
    event_name: str = Field(description="事件名称")
    date: str = Field(description="事件发生日期")

def custom_error_handler(error:Exception)->str:
    """自定义错误处理器"""
    error_str = str(error)
    print(f"捕获到错误类型：{type(error).__name__}")
    print(f"错误详情：{error_str}")
    if isinstance(error, StructuredOutputValidationError):
        return "数据格式有误，请检查字段是否符合要求。"
    elif isinstance(error, MultipleStructuredOutputsError):
        return "检测到多个响应，请选择最相关的一个进行返回。"
    else:
        return f"Error: {error_str}"



agent = create_agent(
model=model,
response_format=ToolStrategy(
    Union[ContactInfo, EventInfo],
    tool_message_content="信息提取成功",
    handle_errors=custom_error_handler
    )
)
response = agent.invoke(
{
"messages": [
HumanMessage("请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：2026-07-15")
]
}
)
rprint(response)

捕获到错误类型：MultipleStructuredOutputsError
错误详情：Model incorrectly returned multiple structured responses (ContactInfo, EventInfo) when only one is expected.
捕获到错误类型：MultipleStructuredOutputsError
错误详情：Model incorrectly returned multiple structured responses (ContactInfo, EventInfo) when only one is expected.
捕获到错误类型：MultipleStructuredOutputsError
错误详情：Model incorrectly returned multiple structured responses (ContactInfo, EventInfo) when only one is expected.
捕获到错误类型：MultipleStructuredOutputsError
错误详情：Model incorrectly returned multiple structured responses (ContactInfo, EventInfo) when only one is expected.
捕获到错误类型：MultipleStructuredOutputsError
错误详情：Model incorrectly returned multiple structured responses (ContactInfo, EventInfo) when only one is expected.


{
    'messages': [
        HumanMessage(
            content='请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：
2026-07-15',
            additional_kwargs={},
            response_metadata={},
            id='f372d3d8-9439-475e-8e61-8d8e29215ba1'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 126,
                    'prompt_tokens': 433,
                    'total_tokens': 559,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 384
                    },
                    'prompt_cache_hit_tokens': 384,
                    'prompt_cache_miss_tokens': 49
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': '4ed2a586-e75c-4064-b458-2fc6a2915040',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019ffabd-422e-70f1-8065-39ab2e1c3214-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '张三', 'email': 'zhang3@atguigu.com', 'phone': ''},
                    'id': 'call_00_ppwWA84V7ApUCQ7a4XUT8618',
                    'type': 'tool_call'
                },
                {
                    'name': 'EventInfo',
                    'args': {'event_name': '公司年会', 'date': '2026-07-15'},
                    'id': 'call_01_0no4C2ijraBvkinBTqBy9914',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 433,
                'output_tokens': 126,
                'total_tokens': 559,
                'input_token_details': {'cache_read': 384},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='检测到多个响应，请选择最相关的一个进行返回。',
            name='ContactInfo',
            id='41407ff4-d64a-4a96-b336-7b29449b24e6',
            tool_call_id='call_00_ppwWA84V7ApUCQ7a4XUT8618'
        ),
        ToolMessage(
            content='检测到多个响应，请选择最相关的一个进行返回。',
            name='EventInfo',
            id='50ec9648-df1a-4ce3-8cd6-f36091731ae1',
            tool_call_id='call_01_0no4C2ijraBvkinBTqBy9914'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 126,
                    'prompt_tokens': 610,
                    'total_tokens': 736,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 512
                    },
                    'prompt_cache_hit_tokens': 512,
                    'prompt_cache_miss_tokens': 98
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': '93233ffd-3f03-4cd7-951d-b260dd7ac3ad',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019ffabd-47b9-77f1-8575-76c27430e23a-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '张三', 'email': 'zhang3@atguigu.com', 'phone': ''},
                    'id': 'call_00_GPUvWb6SenSznfh1TLKq7173',
                    'type': 'tool_call'
     